In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Last run date: {dt.datetime.today()}')

Last run date: 2024-03-11 10:47:43.096596


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: ad_hoc
Subtask: linear_ecnl_transformation


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT\n'
 'bigAccountId,\n'
 'dtmBooking,\n'
 'BookingQuarter,\n'
 'fltNetChgOff,\n'
 'MonthEndDate\n'
 'FROM medusa.riskdb.accountingReports.tblAccounting_LoanCOandNA_ME\n'
 'WHERE \n'
 "MonthEndDate = '2024-02-29' \n"
 "-- AND dtmBooking >= '2013-01-01' \n"
 "-- AND dtmBooking < '2020-01-01'")


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

Wall time: 1.52 s


,bigAccountId,dtmBooking,BookingQuarter,fltNetChgOff,MonthEndDate
0,914657,2012-05-25 10:34:36.000,2012 Q2,11652.01,2024-02-29
1,914668,2012-05-07 17:33:39.000,2012 Q2,8788.29,2024-02-29
2,914681,2012-05-14 08:04:14.000,2012 Q2,-2580.23,2024-02-29
3,914771,2012-05-14 12:48:23.000,2012 Q2,-198.38,2024-02-29
4,914777,2012-05-15 13:54:05.000,2012 Q2,7553.05,2024-02-29
...,...,...,...,...,...
111269,6704534,2023-03-30 14:58:32.433,2023 Q1,13240.17,2024-02-29
111270,6705646,2023-03-24 14:26:29.547,2023 Q1,15572.47,2024-02-29
111271,6705943,2023-04-11 10:51:18.903,2023 Q2,8188.41,2024-02-29
111272,6706216,2023-04-21 15:49:00.763,2023 Q2,8165.90,2024-02-29


### Save

In [8]:
%%time

# save
str_filename = 'df_loss.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 1.31 s


### Upload to s3

In [9]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/01_linear_transformations/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 852 ms


### Clean-up

In [10]:
os.remove(str_local_path)